## Importing libraries

In [1]:
import pandas as pd
import numpy as np

## Loading dataset

In [2]:
df=pd.read_csv("Skycity aukland restaurants & bars.csv")

In [3]:
df

,CuisineType,RestaurantID,RestaurantName,Segment,Subregion,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,InStoreRevenue,...,DeliveryCostPerOrder,SD_DeliveryTotalCost,InStoreNetProfit,UberEatsNetProfit,DoorDashNetProfit,SelfDeliveryNetProfit,InStoreShare,UE_share,DD_share,SD_share
0,Burgers,25731,Urban Burgers House,Cafe,North Shore,1.03,43.97,668,197,8662.09,...,3.25,458.25,3682.14,1352.45,752.78,2177.19,0.42,0.45,0.25,0.3
1,Burgers,25123,Urban Burgers Diner,QSR,South Auckland,1.05,40.45,1388,259,10476.55,...,4.72,1600.08,3605.72,1318.61,731.99,3119.38,0.23,0.45,0.25,0.3
2,Burgers,25177,King Burgers Eatery,Cafe,West Auckland,1.04,40.03,1717,524,20975.72,...,3.25,1163.50,7810.95,1555.90,863.42,4172.99,0.44,0.45,0.25,0.3
3,Burgers,25540,Classic Burgers Tavern,QSR,North Shore,1.03,36.28,1083,216,7836.48,...,0.89,231.40,2546.02,-72.25,-40.20,2833.26,0.25,0.45,0.25,0.3
4,Burgers,25258,Lucky Burgers Bistro,Cafe,South Auckland,1.05,34.34,1230,261,8962.74,...,2.66,774.06,3093.09,226.17,125.53,2674.56,0.27,0.45,0.25,0.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1691,Thai,26412,Lotus Bistro,Ghost Kitchen,North Shore,1.03,45.81,487,31,1420.11,...,5.02,456.82,746.46,2356.69,1416.08,1734.40,0.07,0.50,0.30,0.2
1692,Thai,26275,Dragon Bistro,Ghost Kitchen,North Shore,1.03,44.20,956,62,2740.40,...,5.02,898.58,1338.12,3720.20,2230.45,2964.70,0.07,0.50,0.30,0.2
1693,Thai,26454,Dragon Corner,QSR,South Auckland,1.05,44.72,758,62,2772.64,...,1.48,205.72,969.04,459.09,275.72,1966.80,0.09,0.50,0.30,0.2
1694,Thai,25699,Golden Corner,Cafe,North Shore,1.03,36.98,1084,124,4585.52,...,1.48,284.16,1585.72,1168.16,700.90,2171.15,0.13,0.50,0.30,0.2


## A. Feature engineering

Total revenue

In [4]:
df['TotalRevenue'] = (
    df['InStoreRevenue'] + df['UberEatsRevenue'] +
    df['DoorDashRevenue'] + df['SelfDeliveryRevenue']
)

Total net profit (target variable)

In [5]:
df['TotalNetProfit'] = (
    df['InStoreNetProfit'] + df['UberEatsNetProfit'] +
    df['DoorDashNetProfit'] + df['SelfDeliveryNetProfit']
    )

## Creating model-ready features
### 1. Channel revenue ratio

In [6]:
df['InStore_RevenueRatio']  = df['InStoreRevenue']      / df['TotalRevenue']
df['UE_RevenueRatio']       = df['UberEatsRevenue']     / df['TotalRevenue']
df['DD_RevenueRatio']       = df['DoorDashRevenue']     / df['TotalRevenue']
df['SD_RevenueRatio']       = df['SelfDeliveryRevenue'] / df['TotalRevenue']

### 2. Cost to revenue ratios

In [7]:
df['COGS_to_Revenue']       = df['COGSRate']  * 1           # already a ratio
df['OPEX_to_Revenue']       = df['OPEXRate']  * 1
df['DeliveryCost_to_SDRev'] = np.where(
    df['SelfDeliveryRevenue'] > 0,
    df['SD_DeliveryTotalCost'] / df['SelfDeliveryRevenue'],
    0
)
df['TotalCostRate']         = df['COGSRate'] + df['OPEXRate']

### 3. Profit per order

In [8]:
total_orders_safe = np.where(df['MonthlyOrders'] > 0, df['MonthlyOrders'], 1)
df['ProfitPerOrder']          = df['TotalNetProfit']      / total_orders_safe
df['InStore_ProfitPerOrder']  = df['InStoreNetProfit']    / np.where(df['InStoreOrders']       > 0, df['InStoreOrders'], 1)
df['UE_ProfitPerOrder']       = df['UberEatsNetProfit']   / np.where(df['UberEatsOrders']      > 0, df['UberEatsOrders'], 1)
df['DD_ProfitPerOrder']       = df['DoorDashNetProfit']   / np.where(df['DoorDashOrders']      > 0, df['DoorDashOrders'], 1)
df['SD_ProfitPerOrder']       = df['SelfDeliveryNetProfit'] / np.where(df['SelfDeliveryOrders'] > 0, df['SelfDeliveryOrders'], 1)

### 4. Interaction terms

In [9]:
df['CommRate_x_UEshare']        = df['CommissionRate'] * df['UE_share']
df['DeliveryCost_x_SDshare']    = df['DeliveryCostPerOrder'] * df['SD_share']
df['CommRate_x_DDshare']        = df['CommissionRate'] * df['DD_share']

### 5. Growth adjusted demand features

In [10]:
df['GrowthAdj_Orders']          = df['MonthlyOrders']  * df['GrowthFactor']
df['GrowthAdj_Revenue']         = df['TotalRevenue']   * df['GrowthFactor']
df['GrowthAdj_UEOrders']        = df['UberEatsOrders'] * df['GrowthFactor']
df['GrowthAdj_SDOrders']        = df['SelfDeliveryOrders'] * df['GrowthFactor']

## B. Data processing
### Label encoding

In [11]:
cat_cols = ['CuisineType', 'Segment', 'Subregion']
label_maps = {}
for col in cat_cols:
    cats = sorted(df[col].unique())
    label_maps[col] = {v: i for i, v in enumerate(cats)}
    df[col + '_enc'] = df[col].map(label_maps[col])
    print(f"  {col}: {len(cats)} categories encoded")

  CuisineType: 8 categories encoded
  Segment: 4 categories encoded
  Subregion: 4 categories encoded


### Outlier handling (IQR capping on targets + key numeric)

In [12]:
def iqr_cap(series, factor=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    return series.clip(lower, upper), lower, upper
 
outlier_cols = ['TotalNetProfit', 'InStoreNetProfit', 'UberEatsNetProfit',
                'DoorDashNetProfit', 'SelfDeliveryNetProfit', 'TotalRevenue']
outlier_stats = {}
for col in outlier_cols:
    df[col + '_capped'], lo, hi = iqr_cap(df[col])
    outlier_stats[col] = {'lower': lo, 'upper': hi,
                          'n_clipped': ((df[col] < lo) | (df[col] > hi)).sum()}
 
print(f"\n  Outlier capping (3×IQR):")
for col, s in outlier_stats.items():
    if s['n_clipped'] > 0:
        print(f"    {col}: {s['n_clipped']} rows clipped [{s['lower']:.0f}, {s['upper']:.0f}]")


  Outlier capping (3×IQR):
    InStoreNetProfit: 14 rows clipped [-4054, 8025]
    SelfDeliveryNetProfit: 10 rows clipped [-5754, 9558]


### Feature matrix

In [13]:
feature_cols = [
    'CuisineType_enc', 'Segment_enc', 'Subregion_enc',
    'GrowthFactor', 'AOV',
    'InStore_RevenueRatio', 'UE_RevenueRatio', 'DD_RevenueRatio', 'SD_RevenueRatio',
    'COGSRate', 'OPEXRate', 'CommissionRate',
    'DeliveryRadiusKM', 'DeliveryCostPerOrder',
    'COGS_to_Revenue', 'OPEX_to_Revenue', 'DeliveryCost_to_SDRev', 'TotalCostRate',
    'ProfitPerOrder', 'InStore_ProfitPerOrder', 'UE_ProfitPerOrder',
    'DD_ProfitPerOrder', 'SD_ProfitPerOrder',
    'CommRate_x_UEshare', 'DeliveryCost_x_SDshare', 'CommRate_x_DDshare',
    'GrowthAdj_Orders', 'GrowthAdj_Revenue',
    'GrossMargin', 'OperatingMargin', 'AggregatorRevShare', 'DigitalChannelShare',
    'InStoreShare', 'UE_share', 'DD_share', 'SD_share',
    'MonthlyOrders', 'TotalRevenue'
]
df['GrossMargin'] = 1 - df['COGSRate']
df['OperatingMargin'] = df['TotalNetProfit'] / df['TotalRevenue']
df['AggregatorRevShare'] = (df['UberEatsRevenue'] + df['DoorDashRevenue'] + df['SelfDeliveryRevenue']) / df['TotalRevenue']
df['DigitalChannelShare'] = 1  # All revenue channels sum to total revenue
TARGET       = 'TotalNetProfit_capped'
CHAN_TARGETS = ['InStoreNetProfit_capped', 'UberEatsNetProfit_capped',
                'DoorDashNetProfit_capped', 'SelfDeliveryNetProfit_capped']
 
X = df[feature_cols].values.astype(float)
y = df[TARGET].values.astype(float)

### Manual standardisation

In [14]:
X_mean = X.mean(axis=0)
X_std  = np.where(X.std(axis=0) == 0, 1, X.std(axis=0))
X_sc   = (X - X_mean) / X_std
 
print(f"\n  Feature matrix: {X_sc.shape[0]} rows × {X_sc.shape[1]} features")
print(f"  Target (capped) — mean: ${y.mean():.0f}  std: ${y.std():.0f}\n")


  Feature matrix: 1696 rows × 38 features
  Target (capped) — mean: $4638  std: $5828



### Train test/ split

In [15]:
np.random.seed(42)
idx   = np.random.permutation(len(X_sc))
split = int(0.8 * len(idx))
tr, te = idx[:split], idx[split:]
X_tr, X_te = X_sc[tr], X_sc[te]
y_tr, y_te = y[tr], y[te]
print(f"  Train: {len(tr)}  |  Test: {len(te)}\n")

  Train: 1356  |  Test: 340



## C. Model development
Utility

In [16]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))
 
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot if ss_tot != 0 else 0
 
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))
 
def evaluate(name, y_true, y_pred):
    return {
        'Model': name,
        'RMSE':  rmse(y_true, y_pred),
        'R2':    r2(y_true, y_pred),
        'MAE':   mae(y_true, y_pred)
    }

#### Model 1: Linear regression

In [17]:
print("\n  [1/3] Linear Regression (Ridge, λ=1e-3) …")
 
def linear_regression_train(X, y, lam=1e-3):
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    A  = Xb.T @ Xb + lam * np.eye(Xb.shape[1])
    b  = Xb.T @ y
    return np.linalg.solve(A, b)
 
def linear_regression_predict(X, w):
    Xb = np.hstack([np.ones((X.shape[0], 1)), X])
    return Xb @ w
 
lr_w          = linear_regression_train(X_tr, y_tr)
lr_pred_tr    = linear_regression_predict(X_tr, lr_w)
lr_pred_te    = linear_regression_predict(X_te, lr_w)
lr_pred_all   = linear_regression_predict(X_sc, lr_w)
 
print(f"     Train R²={r2(y_tr, lr_pred_tr):.4f}  |  Test R²={r2(y_te, lr_pred_te):.4f}")


  [1/3] Linear Regression (Ridge, λ=1e-3) …
     Train R²=0.9040  |  Test R²=0.9036


Feature coefficients 

In [18]:
coef = pd.Series(lr_w[1:], index=feature_cols).abs().sort_values(ascending=False)
print("     Top-10 linear predictors:")
for feat, val in coef.head(10).items():
    print(f"       {feat:<35s}  coef_abs={val:.4f}")

     Top-10 linear predictors:
       DeliveryRadiusKM                     coef_abs=26108.9117
       DeliveryCostPerOrder                 coef_abs=24012.3021
       TotalRevenue                         coef_abs=10735.9547
       OperatingMargin                      coef_abs=9477.8985
       MonthlyOrders                        coef_abs=8704.7679
       ProfitPerOrder                       coef_abs=7749.6196
       SD_ProfitPerOrder                    coef_abs=7373.4144
       GrowthAdj_Revenue                    coef_abs=6575.6998
       GrowthAdj_Orders                     coef_abs=6334.4078
       UE_ProfitPerOrder                    coef_abs=4974.3933


#### Model 2: Random forest regressor

In [19]:
class DecisionTreeNode:
    __slots__ = ('feat', 'thresh', 'left', 'right', 'value')
    def __init__(self):
        self.feat = self.thresh = self.left = self.right = self.value = None

In [20]:
def _mse_split(y):
    return np.mean((y - np.mean(y)) ** 2) if len(y) > 0 else 0
 
def _best_split(X, y, n_features):
    best_gain, best_feat, best_thresh = -np.inf, None, None
    base_mse = _mse_split(y)
    feat_idx = np.random.choice(X.shape[1], n_features, replace=False)
    for f in feat_idx:
        vals    = X[:, f]
        threshs = np.unique(np.percentile(vals, np.linspace(10, 90, 10)))
        for t in threshs:
            lm = vals <= t
            if lm.sum() < 2 or (~lm).sum() < 2:
                continue
            gain = base_mse - (lm.sum() * _mse_split(y[lm]) +
                               (~lm).sum() * _mse_split(y[~lm])) / len(y)
            if gain > best_gain:
                best_gain, best_feat, best_thresh = gain, f, t
    return best_feat, best_thresh

In [21]:
def _build_tree(X, y, max_depth, min_samples, n_features, depth=0):
    node = DecisionTreeNode()
    if depth == max_depth or len(y) <= min_samples or len(np.unique(y)) == 1:
        node.value = np.mean(y)
        return node
    feat, thresh = _best_split(X, y, n_features)
    if feat is None:
        node.value = np.mean(y)
        return node
    lm = X[:, feat] <= thresh
    node.feat, node.thresh = feat, thresh
    node.left  = _build_tree(X[lm],  y[lm],  max_depth, min_samples, n_features, depth+1)
    node.right = _build_tree(X[~lm], y[~lm], max_depth, min_samples, n_features, depth+1)
    return node

In [22]:
def _predict_one(node, x):
    while node.value is None:
        node = node.left if x[node.feat] <= node.thresh else node.right
    return node.value

In [23]:
def rf_train(X, y, n_trees=50, max_depth=6, min_samples=5):
    trees = []
    n_features = max(1, int(np.sqrt(X.shape[1])))
    for _ in range(n_trees):
        bi = np.random.choice(len(X), len(X), replace=True)
        trees.append(_build_tree(X[bi], y[bi], max_depth, min_samples, n_features))
    return trees

In [24]:
def rf_predict(trees, X):
    preds = np.array([[_predict_one(t, x) for t in trees] for x in X])
    return preds.mean(axis=1)
 
rf_trees      = rf_train(X_tr, y_tr, n_trees=50, max_depth=6)
rf_pred_tr    = rf_predict(rf_trees, X_tr)
rf_pred_te    = rf_predict(rf_trees, X_te)
rf_pred_all   = rf_predict(rf_trees, X_sc)
 
print(f"     Train R²={r2(y_tr, rf_pred_tr):.4f}  |  Test R²={r2(y_te, rf_pred_te):.4f}")

     Train R²=0.9845  |  Test R²=0.9786


### Feature importance

In [25]:
def _feature_importances(trees, n_features):
    scores = np.zeros(n_features)
    for tree in trees:
        stack = [tree]
        while stack:
            node = stack.pop()
            if node.value is None:
                scores[node.feat] += 1
                stack += [node.left, node.right]
    return scores / scores.sum()
 
rf_imp = pd.Series(_feature_importances(rf_trees, X_sc.shape[1]), index=feature_cols)
rf_imp_sorted = rf_imp.sort_values(ascending=False)
print("     Top-10 RF feature importances:")
for feat, val in rf_imp_sorted.head(10).items():
    print(f"       {feat:<35s}  importance={val:.4f}")

     Top-10 RF feature importances:
       ProfitPerOrder                       importance=0.1031
       OperatingMargin                      importance=0.0964
       GrowthAdj_Revenue                    importance=0.0684
       TotalRevenue                         importance=0.0677
       GrowthAdj_Orders                     importance=0.0590
       MonthlyOrders                        importance=0.0580
       DD_ProfitPerOrder                    importance=0.0478
       UE_ProfitPerOrder                    importance=0.0438
       TotalCostRate                        importance=0.0404
       SD_ProfitPerOrder                    importance=0.0330


### Model 3: Gradient boosting regressor

In [26]:
print("\n  [3/3] Gradient Boosting Regressor (100 trees, lr=0.1, depth=4) …")
 
def gb_train(X, y, n_trees=100, lr=0.1, max_depth=4, min_samples=5):
    init_pred  = np.full(len(y), np.mean(y))
    trees, init = [], np.mean(y)
    residuals  = y - init_pred
    n_features = max(1, int(np.sqrt(X.shape[1])))
    for _ in range(n_trees):
        tree = _build_tree(X, residuals, max_depth, min_samples, n_features)
        upd  = np.array([_predict_one(tree, x) for x in X])
        residuals -= lr * upd
        trees.append(tree)
    return init, trees, lr


  [3/3] Gradient Boosting Regressor (100 trees, lr=0.1, depth=4) …


In [27]:
def gb_predict(init, trees, lr, X):
    pred = np.full(len(X), init)
    for tree in trees:
        pred += lr * np.array([_predict_one(tree, x) for x in X])
    return pred
 
gb_init, gb_trees, gb_lr = gb_train(X_tr, y_tr, n_trees=100, lr=0.1, max_depth=4)
gb_pred_tr  = gb_predict(gb_init, gb_trees, gb_lr, X_tr)
gb_pred_te  = gb_predict(gb_init, gb_trees, gb_lr, X_te)
gb_pred_all = gb_predict(gb_init, gb_trees, gb_lr, X_sc)
 
print(f"     Train R²={r2(y_tr, gb_pred_tr):.4f}  |  Test R²={r2(y_te, gb_pred_te):.4f}")

     Train R²=0.9979  |  Test R²=0.9931


#### Ensemble: average of all three models

In [28]:
ens_pred_te  = (lr_pred_te  + rf_pred_te  + gb_pred_te)  / 3
ens_pred_all = (lr_pred_all + rf_pred_all + gb_pred_all) / 3

## D. Model evaluation

In [29]:
results = [
    evaluate('Linear Regression',   y_te, lr_pred_te),
    evaluate('Random Forest',        y_te, rf_pred_te),
    evaluate('Gradient Boosting',    y_te, gb_pred_te),
    evaluate('Ensemble (avg)',        y_te, ens_pred_te),
]
eval_df = pd.DataFrame(results).set_index('Model')
eval_df['RMSE_$'] = eval_df['RMSE'].round(2)
eval_df['R2_%']   = (eval_df['R2'] * 100).round(2)
eval_df['MAE_$']  = eval_df['MAE'].round(2)

### Sensitivity / Peturbation analysis

In [30]:
baseline_rmse = rmse(y_te, gb_pred_te)
sens_scores = {}
for i, feat in enumerate(feature_cols):
    X_pert = X_te.copy()
    X_pert[:, i] = X_pert[:, i] + X_pert[:, i].std() * 0.1   # +10% std shock
    pred_pert = gb_predict(gb_init, gb_trees, gb_lr, X_pert)
    sens_scores[feat] = abs(rmse(y_te, pred_pert) - baseline_rmse)

In [31]:
sens_df = pd.Series(sens_scores).sort_values(ascending=False)
print("  Top-10 sensitive features:")
for feat, score in sens_df.head(10).items():
    bar = '█' * int(score / sens_df.max() * 20)

  Top-10 sensitive features:


### Select best model for downstream use

In [32]:
best_model_name = eval_df['R2_%'].idxmax()
if best_model_name == 'Linear Regression':
    best_pred_all = lr_pred_all
elif best_model_name == 'Random Forest':
    best_pred_all = rf_pred_all
elif best_model_name == 'Gradient Boosting':
    best_pred_all = gb_pred_all
else:
    best_pred_all = ens_pred_all

df['PredictedNetProfit'] = best_pred_all


### Scenario simulation engine
Simulate

In [33]:
def predict_scenario(df_mod):
    """Recompute features and predict using the best model."""
    d = df_mod.copy()
 
    # Recompute derived columns
    d['TotalRevenue'] = (d['InStoreRevenue'] + d['UberEatsRevenue'] +
                         d['DoorDashRevenue'] + d['SelfDeliveryRevenue'])
    for col in cat_cols:
        d[col + '_enc'] = d[col].map(label_maps[col]).fillna(0)
 
    safe_rev = np.where(d['TotalRevenue'] > 0, d['TotalRevenue'], 1)
    d['InStore_RevenueRatio']  = d['InStoreRevenue']      / safe_rev
    d['UE_RevenueRatio']       = d['UberEatsRevenue']     / safe_rev
    d['DD_RevenueRatio']       = d['DoorDashRevenue']     / safe_rev
    d['SD_RevenueRatio']       = d['SelfDeliveryRevenue'] / safe_rev
    d['COGS_to_Revenue']       = d['COGSRate']
    d['OPEX_to_Revenue']       = d['OPEXRate']
    sd_rev_safe = np.where(d['SelfDeliveryRevenue'] > 0, d['SelfDeliveryRevenue'], 1)
    d['DeliveryCost_to_SDRev'] = d['SD_DeliveryTotalCost'] / sd_rev_safe
    d['TotalCostRate']         = d['COGSRate'] + d['OPEXRate']
 
    safe_ord = np.where(d['MonthlyOrders'] > 0, d['MonthlyOrders'], 1)
    d['TotalNetProfit'] = (d['InStoreNetProfit'] + d['UberEatsNetProfit'] +
                           d['DoorDashNetProfit'] + d['SelfDeliveryNetProfit'])
    d['ProfitPerOrder']         = d['TotalNetProfit'] / safe_ord
    d['InStore_ProfitPerOrder'] = d['InStoreNetProfit']    / np.where(d['InStoreOrders']       > 0, d['InStoreOrders'], 1)
    d['UE_ProfitPerOrder']      = d['UberEatsNetProfit']   / np.where(d['UberEatsOrders']      > 0, d['UberEatsOrders'], 1)
    d['DD_ProfitPerOrder']      = d['DoorDashNetProfit']   / np.where(d['DoorDashOrders']      > 0, d['DoorDashOrders'], 1)
    d['SD_ProfitPerOrder']      = d['SelfDeliveryNetProfit'] / np.where(d['SelfDeliveryOrders'] > 0, d['SelfDeliveryOrders'], 1)
 
    d['CommRate_x_UEshare']     = d['CommissionRate'] * d['UE_share']
    d['DeliveryCost_x_SDshare'] = d['DeliveryCostPerOrder'] * d['SD_share']
    d['CommRate_x_DDshare']     = d['CommissionRate'] * d['DD_share']
    d['GrowthAdj_Orders']       = d['MonthlyOrders']  * d['GrowthFactor']
    d['GrowthAdj_Revenue']      = d['TotalRevenue']   * d['GrowthFactor']
    d['GrowthAdj_UEOrders']     = d['UberEatsOrders'] * d['GrowthFactor']
    d['GrowthAdj_SDOrders']     = d['SelfDeliveryOrders'] * d['GrowthFactor']
    d['GrossMargin']            = 1 - d['COGSRate']
    d['OperatingMargin']        = d['TotalNetProfit'] / safe_rev
    d['AggregatorRevShare']     = d['UE_RevenueRatio'] + d['DD_RevenueRatio']
    d['DigitalChannelShare']    = d['UE_share'] + d['DD_share'] + d['SD_share']
 
    Xm     = d[feature_cols].values.astype(float)
    Xm_sc  = (Xm - X_mean) / X_std
    return gb_predict(gb_init, gb_trees, gb_lr, Xm_sc)
 
baseline_profit = df['PredictedNetProfit'].sum()
print(f"\n  Baseline Total Predicted Net Profit: ${baseline_profit:,.0f}\n")
 
scenarios = {}


  Baseline Total Predicted Net Profit: $7,877,668



scenario 1: Increased Aggregator Reliance (+15% UE & DD share)

In [34]:
s1 = df.copy()
shift = 0.15
s1['UE_share'] = (s1['UE_share'] + shift / 2).clip(0, 1)
s1['DD_share'] = (s1['DD_share'] + shift / 2).clip(0, 1)
s1['InStoreShare'] = (1 - s1['UE_share'] - s1['DD_share'] - s1['SD_share']).clip(0, 1)

Redistribute revenue proportionally

In [35]:
agg_uplift = 1 + shift
s1['UberEatsRevenue']  *= agg_uplift
s1['DoorDashRevenue']  *= agg_uplift
s1['InStoreRevenue']   *= (1 - shift)
s1_profit = predict_scenario(s1).sum()
scenarios['S1: +15% Aggregator Reliance'] = s1_profit

Scenario 2: Decreased Aggregator Reliance (−15%)

In [36]:
s2 = df.copy()
s2['UE_share'] = (s2['UE_share'] - shift / 2).clip(0, 1)
s2['DD_share'] = (s2['DD_share'] - shift / 2).clip(0, 1)
s2['InStoreShare'] = (1 - s2['UE_share'] - s2['DD_share'] - s2['SD_share']).clip(0, 1)
s2['UberEatsRevenue'] *= (1 - shift)
s2['DoorDashRevenue'] *= (1 - shift)
s2['InStoreRevenue']  *= (1 + shift)
s2_profit = predict_scenario(s2).sum()
scenarios['S2: −15% Aggregator Reliance'] = s2_profit

Scenario 3: Commission Rate +5% (aggregators raise fees)

In [37]:
s3 = df.copy()
s3['CommissionRate'] = (s3['CommissionRate'] + 0.05).clip(0, 0.5)
s3_profit = predict_scenario(s3).sum()
scenarios['S3: Commission Rate +5pp'] = s3_profit


Scenario 4: Commission Rate −5% (negotiated reduction)

In [38]:
s4 = df.copy()
s4['CommissionRate'] = (s4['CommissionRate'] - 0.05).clip(0, 0.5)
s4_profit = predict_scenario(s4).sum()
scenarios['S4: Commission Rate −5pp'] = s4_profit

Scenario 5: Self-Delivery Radius Expansion (+5 km)

In [39]:
s5 = df.copy()
s5['DeliveryRadiusKM'] = s5['DeliveryRadiusKM'] + 5
s5['SelfDeliveryOrders'] = (s5['SelfDeliveryOrders'] * 1.20).astype(int)
s5['SelfDeliveryRevenue'] *= 1.20
s5['SD_DeliveryTotalCost'] *= 1.30   # cost grows faster than orders
s5['SD_share'] = (s5['SD_share'] + 0.05).clip(0, 1)
s5_profit = predict_scenario(s5).sum()
scenarios['S5: Self-Delivery Radius +5km'] = s5_profit

Scenario 6: Demand Growth +10% (GrowthFactor bump)

In [40]:
s6 = df.copy()
s6['GrowthFactor'] = s6['GrowthFactor'] * 1.10
s6['MonthlyOrders'] = (s6['MonthlyOrders'] * 1.10).astype(int)
s6['InStoreRevenue']   *= 1.10
s6['UberEatsRevenue']  *= 1.10
s6['DoorDashRevenue']  *= 1.10
s6['SelfDeliveryRevenue'] *= 1.10
s6_profit = predict_scenario(s6).sum()
scenarios['S6: +10% Demand Growth']  = s6_profit

Scenario 7: Margin Erosion — COGS +5% 

In [41]:
s7 = df.copy()
s7['COGSRate'] = (s7['COGSRate'] + 0.05).clip(0, 0.95)
s7_profit = predict_scenario(s7).sum()
scenarios['S7: COGS Rate +5pp (erosion)'] = s7_profit

In [42]:
df.sort_index(inplace=True)

### Prescriptive optimizing layer
Optimizing profit maximising channel mix

### Key performance indicators
predictive net profit

In [43]:
df['PredictedNetProfit'] = df['TotalNetProfit'] * df['GrowthFactor']

Profit sensitivity index(cost vulnerability)

In [44]:
df['TotalCostRate'] = df['COGSRate'] + df['OPEXRate'] + df['CommissionRate']

df['ProfitSensitivityIndex'] = df['TotalCostRate'] / (df['TotalNetProfit'] + 1e-6)

Channel mix efficiency score

In [45]:
df['ChannelMixEfficiency'] = (
    df['InStoreNetProfit'] * df['InStoreShare'] +
    df['UberEatsNetProfit'] * df['UE_share'] +
    df['DoorDashNetProfit'] * df['DD_share'] +
    df['SelfDeliveryNetProfit'] * df['SD_share']
) / (df['TotalNetProfit'] + 1e-6)

Break even commision rate

In [46]:
df['BreakEvenCommissionRate'] = (
    df['CommissionRate'] * (
        (df['UberEatsNetProfit'] + df['DoorDashNetProfit']) /
        ((df['UberEatsRevenue'] + df['DoorDashRevenue']) + 1e-6)
    )
)

Optimizing uplift %

In [47]:
df['OptimizedProfit'] = (
    df['InStoreNetProfit'] +
    (df['UberEatsNetProfit'] * 0.9) +
    (df['DoorDashNetProfit'] * 0.9) +
    (df['SelfDeliveryNetProfit'] * 1.1)
)

df['OptimizationUplift_pct'] = (
    (df['OptimizedProfit'] - df['TotalNetProfit']) /
    (df['TotalNetProfit'] + 1e-6)
) * 100

Kpi summary table

In [ ]:
kpi_summary = df[[
    'PredictedNetProfit',
    'ProfitSensitivityIndex',
    'ChannelMixEfficiency',
    'BreakEvenCommissionRate',
    'OptimizationUplift_pct'
]].describe()

In [49]:
print(kpi_summary)

       PredictedNetProfit  ProfitSensitivityIndex  ChannelMixEfficiency  \
count         1696.000000             1696.000000           1696.000000   
mean          4776.812321                0.000038              0.338720   
std           6012.019995                0.001067              0.290002   
min         -10702.471500               -0.022146             -6.200765   
25%            572.502975                0.000046              0.262253   
50%           5078.911000                0.000109              0.301817   
75%           8582.913000                0.000192              0.372046   
max          28463.094400                0.013471              4.160752   

       BreakEvenCommissionRate  OptimizationUplift_pct  
count              1696.000000             1696.000000  
mean                  0.002122               -2.467845  
std                   0.036213               29.116234  
min                  -0.083076             -448.224397  
25%                  -0.033133         